In [2]:
from datasets import load_dataset
import json

ds = load_dataset('xiaozheyao/megachat', name='sharegpt', split='train')


/home/fireblue/miniconda3/envs/download/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds[0]

{'conversations': [{'role': 'user',
   'content': "Summarize the main ideas of Jeff Walker's Product Launch Formula into bullet points as it pertains to a growth marketing agency implementing these strategies and tactics for their clients..."},
  {'role': 'assistant',
   'content': "Here are the main ideas of Jeff Walker's Product Launch Formula that can be applied by a growth marketing agency for their clients:\n\n1. Identify the target audience and their needs: Understand the ideal customer for the product or service, and create a messaging that resonates with them.\n2. Pre-launch: Build anticipation and excitement for the launch by creating buzz, gathering testimonials and case studies, and using social media to create awareness.\n3. Launch: Use a well-crafted launch sequence to maximize sales and conversions. This can include offering bonuses, creating scarcity, and using a deadline to create urgency.\n4. Post-launch: Follow up with customers, gather feedback, and continue to provi

In [4]:
from pathlib import Path
import json


def is_task_oriented(conversations):
    first_user = next((m for m in conversations if m["role"] == "user"), None)
    if first_user is None:
        return False
    q = first_user["content"].lower()

    keywords = [
        "how do i", "how to", "step by step", "help me",
        "guide me", "set up", "install", "open ",
        "configure", "use ", "create a new", "click",
        "navigate", "settings"
    ]
    bad_keywords = ["summarize", "summary of", "explain", "what are the main ideas"]

    if any(b in q for b in bad_keywords):
        return False
    return any(k in q for k in keywords)



folder = Path('../megachat_gpt4o/')
folder.mkdir(exist_ok=True, parents=True)

with open(folder/'chat.jsonl', 'w') as f:
    kept = 0
    for idx, item in enumerate(ds):
        conv = item["conversations"]
        
        if not is_task_oriented(conv):
            continue
        
        sample = {
            "id": f"text_instruct_megachat_gpt4o_{kept:06d}",
            "task_type": "text_instruct",
            "images": [],
            "bboxes": [],
            "grounding_targets": [],
            "conversations": [
                {"from": c["role"], "value": c["content"]}
                for c in conv
            ],
            "meta": {
                "src_dataset": "megachat_gpt4o",
                "source": item.get("meta", {}).get("source")
            },
        }
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")
        kept += 1

print(f"Finished writing {kept} samples to {folder/'chat.jsonl'}")

Finished writing 13809 samples to ../megachat_gpt4o/chat.jsonl
